# Classificação com TensorFlow/Keras em Python

Este notebook apresenta em células as mesmas etapas de `index.js` e de `VERSAO-PYTHON.md`: preparar os dados, montar e treinar uma rede neural e prever uma categoria.

> **Importante:** o exemplo treina com apenas três registros fictícios. É didático e não produz previsões confiáveis para decisões reais.

## Preparação

Selecione um kernel Python com uma versão compatível com o TensorFlow disponível no seu sistema. Se TensorFlow ainda não estiver instalado nesse ambiente, execute no terminal `python -m pip install tensorflow` ou, em uma célula, `%pip install tensorflow`. Depois, reinicie o kernel se necessário.

## 1. Importar TensorFlow

O alias `tf` será usado para acessar TensorFlow e sua API Keras.

In [ ]:
import tensorflow as tf

## 2. Definir as funções de treinamento e previsão

`train_model` cria uma rede com sete entradas, uma camada oculta de 80 unidades e três saídas. `relu` é a ativação da camada oculta e `softmax` transforma as saídas em probabilidades.

O treinamento usa Adam, `categorical_crossentropy` e 100 épocas. O callback imprime a perda ao final de cada época. A função `predict` calcula as probabilidades de um lote e associa cada probabilidade ao índice da categoria.

In [ ]:
def train_model(input_xs, output_ys):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(7,)),
        tf.keras.layers.Dense(80, activation="relu"),
        tf.keras.layers.Dense(3, activation="softmax"),
    ])

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    class LossPrinter(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            logs = logs or {}
            print(f"Epoch: {epoch}: loss = {logs.get('loss')}")

    model.fit(
        input_xs,
        output_ys,
        epochs=100,
        shuffle=True,
        verbose=0,
        callbacks=[LossPrinter()],
    )

    return model


def predict(model, pessoa):
    pred_array = model.predict(pessoa, verbose=0)
    return [
        {"prob": float(prob), "index": index}
        for index, prob in enumerate(pred_array[0])
    ]

## 3. Preparar os exemplos e seus rótulos

Cada linha de entrada contém, nesta ordem: idade normalizada, indicadores de cor (azul, vermelho, verde) e indicadores de localização (São Paulo, Rio, Curitiba). Nas codificações one-hot, `1` indica a opção selecionada e `0` as demais.

Os rótulos têm a ordem `premium`, `medium`, `basic`: `[1, 0, 0]`, `[0, 1, 0]` e `[0, 0, 1]`. As linhas de entrada e os rótulos na mesma posição formam um exemplo de treinamento.

In [ ]:
# Ordem: idade_normalizada, azul, vermelho, verde, São Paulo, Rio, Curitiba
tensor_pessoas_normalizado = [
    [0.33, 1, 0, 0, 1, 0, 0],  # Erick: azul, São Paulo
    [0,    0, 1, 0, 0, 1, 0],  # Ana: vermelha, Rio
    [1,    0, 0, 1, 0, 0, 1],  # Carlos: verde, Curitiba
]

labels_nomes = ["premium", "medium", "basic"]
tensor_labels = [
    [1, 0, 0],  # Erick: premium
    [0, 1, 0],  # Ana: medium
    [0, 0, 1],  # Carlos: basic
]

input_xs = tf.convert_to_tensor(tensor_pessoas_normalizado, dtype=tf.float32)
output_ys = tf.convert_to_tensor(tensor_labels, dtype=tf.float32)

print("Formato das entradas:", input_xs.shape)
print("Formato dos rótulos:", output_ys.shape)

## 4. Treinar o modelo

Esta célula executa o treinamento. Ela imprime a perda em cada época; como são 100 épocas, haverá 100 linhas de acompanhamento.

In [ ]:
model = train_model(input_xs, output_ys)

## 5. Preparar a pessoa para classificar

O dicionário descreve Zé como verde e de Curitiba. Porém, o vetor abaixo reproduz o vetor atual de `index.js`: `[0.2, 1, 0, 0, 1, 0, 0]`, que representa azul e São Paulo. O modelo recebe o vetor numérico, não o dicionário.

Para classificar a descrição de Zé (verde e Curitiba), use a alternativa comentada `[0.2, 0, 0, 1, 0, 0, 1]`. A idade `0.2` vem de `(28 - 25) / (40 - 25)`. A ordem das sete características deve ser a mesma usada no treinamento.

In [ ]:
pessoa = {
    "nome": "zé",
    "idade": 28,
    "cor": "verde",
    "localizacao": "Curitiba",
}

# Reproduz o vetor usado atualmente em index.js: azul e São Paulo.
pessoa_tensor_normalizado = [
    [0.2, 1, 0, 0, 1, 0, 0]
]

# Para representar a descrição de Zé, substitua pelo vetor abaixo:
# pessoa_tensor_normalizado = [
#     [0.2, 0, 0, 1, 0, 0, 1]
# ]

## 6. Prever, ordenar e exibir

`predict` calcula uma probabilidade para cada categoria. A seguir, as categorias são ordenadas da maior para a menor probabilidade, os índices são convertidos nos nomes definidos em `labels_nomes` e os valores são apresentados como porcentagens com duas casas decimais.

A porcentagem mais alta indica a saída mais provável do modelo para aquele vetor, não uma garantia. Com somente três exemplos de treinamento, o resultado é apenas demonstrativo.

In [ ]:
predictions = predict(model, pessoa_tensor_normalizado)

results = sorted(predictions, key=lambda item: item["prob"], reverse=True)
for item in results:
    categoria = labels_nomes[item["index"]]
    porcentagem = item["prob"] * 100
    print(f"{categoria} ({porcentagem:.2f}%)")

## 7. Correspondência com JavaScript

| TensorFlow.js | TensorFlow/Keras em Python | Função |
|---|---|---|
| `tf.sequential()` | `tf.keras.Sequential(...)` | Cria o modelo sequencial |
| `tf.layers.dense(...)` | `tf.keras.layers.Dense(...)` | Cria uma camada densa |
| `model.compile(...)` | `model.compile(...)` | Configura o treinamento |
| `model.fit(...)` | `model.fit(...)` | Treina o modelo |
| `tf.tensor2d(...)` | `tf.convert_to_tensor(...)` | Converte os dados em tensores |
| `model.predict(...)` | `model.predict(...)` | Calcula as previsões |
| `onEpochEnd` | `Callback.on_epoch_end` | Executa código ao fim de cada época |

As porcentagens podem variar entre execuções e bibliotecas por diferenças na inicialização dos parâmetros e nos detalhes de execução. O objetivo é comparar o fluxo e os conceitos, não obter números idênticos.